# Liver & Tumor Segmentation — Training Notebook

**Based on:** FasNet Paper (Nature Scientific Reports, 2025)  
**Dataset:** LiTS17 (Kaggle PNG version by andrewmvd)  
**Architecture:** Standard 2D U-Net  
**Target:** Dice > 0.70 (liver), > 0.50 (tumor) on first run

---

### Cell 1 — Verify GPU

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
# If GPU not available, go to Runtime > Change runtime type > GPU

### Cell 2 — Dataset Path Setup

On **Kaggle**: dataset is at `/kaggle/input/lits-png/`  
On **Colab**: mount Drive and set your path accordingly

In [ ]:
import os

# ========== KAGGLE ==========
DATASET_ROOT = "/kaggle/input/lits-png"
TRAIN_CSV    = f"{DATASET_ROOT}/lits_train.csv"
VAL_CSV      = f"{DATASET_ROOT}/lits_probe.csv"

# Verify files are accessible
if os.path.exists(DATASET_ROOT):
    print("Dataset found!")
    print(os.listdir(DATASET_ROOT)[:10])
else:
    print(f"Dataset NOT found at {DATASET_ROOT}")
    print("Make sure you added the 'lits-png' dataset to this notebook")

### Cell 3 — Install Dependencies & Import Modules

In [ ]:
!pip install tqdm pandas Pillow opencv-python-headless -q

import sys
sys.path.insert(0, '/kaggle/working')  # ensure project root is in path

from src.models.unet import UNet
from src.training.loss import CombinedLoss
from src.training.metrics import dice_score
from src.data.dataset_loader import LITSDataset

print("All modules imported successfully ✓")

### Cell 4 — Sanity Check Before Training

In [ ]:
# Test that model produces correct output shape
model = UNet(in_channels=1, out_channels=2)
dummy = torch.randn(2, 1, 256, 256)
out   = model(dummy)
print(f"Input:  {dummy.shape}")
print(f"Output: {out.shape}")
assert out.shape == (2, 2, 256, 256)
print("Model OK — proceeding to training ✓")
del model, dummy, out  # free memory

### Cell 5 — Run Training

**Config (FasNet paper-proven):**
- Batch size: 32
- Epochs: 100 (stabilizes ~epoch 60)
- Optimizer: Adam (Dice 0.8766 >> SGD 0.7913)
- Loss: 0.5 BCE + 0.5 Dice
- LR: 1e-4 with ReduceLROnPlateau

In [ ]:
!python -m src.training.train

### Cell 6 — Visualize Training Curves

In [ ]:
from src.utils.visualize import plot_training_curves
from IPython.display import Image, display

log_path = 'results/training_log.csv'
if os.path.exists(log_path):
    plot_training_curves(log_path, save_path='results/training_curves.png')
    display(Image(filename='results/training_curves.png'))
else:
    print(f"Log file not found at {log_path}. Ensure training completed.")

### Cell 7 — Interactive Prediction View

Visualize the model output on validation samples after training completes.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from src.data.dataset_loader import LITSDataset
from src.models.unet import UNet
from IPython.display import Image, display

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(in_channels=1, out_channels=2).to(device)

checkpoint_path = 'models/best_model.pth'
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f"Loaded best model (epoch {checkpoint.get('epoch', '?')}, val_loss {checkpoint.get('val_loss', '?'):.4f})")
    
    # Get sample from validation set
    val_dataset = LITSDataset('data/lits_probe.csv', root_dir='data', apply_clahe=True)
    idx = 42  # Change to view different samples
    image, true_mask = val_dataset[idx]
    
    with torch.no_grad():
        logits = model(image.unsqueeze(0).to(device))
        probs = torch.sigmoid(logits)
        pred_mask = (probs > 0.5).float().squeeze(0).cpu()
    
    # 5-panel plot: CT | True Liver | True Tumor | Pred Liver | Pred Tumor
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    axes[0].imshow(image[0].numpy(), cmap='gray'); axes[0].set_title('CT (CLAHE)'); axes[0].axis('off')
    axes[1].imshow(true_mask[0].numpy(), cmap='gray'); axes[1].set_title('True Liver'); axes[1].axis('off')
    axes[2].imshow(true_mask[1].numpy(), cmap='gray'); axes[2].set_title('True Tumor'); axes[2].axis('off')
    axes[3].imshow(pred_mask[0].numpy(), cmap='gray'); axes[3].set_title('Pred Liver'); axes[3].axis('off')
    axes[4].imshow(pred_mask[1].numpy(), cmap='gray'); axes[4].set_title('Pred Tumor'); axes[4].axis('off')
    plt.tight_layout()
    plt.savefig('results/sample_pred.png', dpi=150)
    plt.show()
else:
    print(f"No checkpoint found at {checkpoint_path}. Train first!")

### Target Metrics (FasNet Paper Reference)

| Stage | Dice Liver | Dice Tumor | What it means |
|---|---|---|---|
| Your first run (base U-Net) | > 0.60 | > 0.30 | Model is learning |
| Good base U-Net | ~0.74 | ~0.50 | Comparable to simple baselines |
| After adding attention | ~0.85 | ~0.70 | Paper-level improvement |
| FasNet (ResNet50+VGG16) | 0.8766 | — | State of the art on this dataset |